# Lasso: total match corners, end to end

## Goal
Load every available league for **22/23, 23/24 and 24/25**, construct past-only corner features, train Lasso on the first two seasons, and evaluate total corners on **24/25**. One model row represents one match; the target is home corners + away corners.

**Run All** executes the configured experiment and displays the combined report here. Start with the settings below; optional search, feature selection, ratings, warm-up and deployment refitting are explicit switches. The notebook does not use the UI recipe layer.

Delivered without real experiment outputs. Bounded verification scope is recorded in [the companion guide](../docs/analytics/lasso_total_corners.md).

## Setup
Select the project's `misc314_py314` kernel. No packages are installed by this notebook. The root is detected whether Jupyter starts in the repository or its `notebooks` folder.

In [ ]:
from pathlib import Path
from functools import partial
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "xdiyo_analytics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the xDiyo repository.")
sys.path.insert(0, str(ROOT / "src"))

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from xdiyo_analytics.data import load_seasons, select_stats, select_prediction_fixtures
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import (
    Stat, ForAgainst, Lag, RollingMean, RollingStd, RollingZScore, EMA,
    NormalizedStanding, Rating, WarmStart, SeededEMA, Hard, evaluate_features,
)
from xdiyo_analytics.ratings import Glicko2, GlickoTransition, build_ratings
from xdiyo_analytics.labels import MatchTotal, create_labels
from xdiyo_analytics.datasets import assemble_dataset
from xdiyo_analytics.splits import TemporalSplit, SplitPlan, create_split_plan
from xdiyo_analytics.analysis import PreTrainingAnalysis, PostTrainingAnalysis
from xdiyo_analytics.reporting import (
    FeatureDistributionReporter, CorrelationAnalysis, FeatureTimeline,
    TopKCorrelationSelector, PerformanceReporter, PredictionDistributionReporter,
    ResidualAnalysisReporter, PredictionTimelineReporter, MatchResultReporter,
    CoefficientReporter, ExperimentLeaderboardReporter, TeamCatalog,
)
from xdiyo_analytics.selection import Candidate, GridCandidates, ModelSelection
from xdiyo_analytics.training import EstimatorAdapter, ExecutionPolicy, save_model, load_model
from xdiyo_analytics.experiments import FootballExperiment, PreparedExperiment, RefitPolicy

## 1. Configure the experiment
`LEAGUES=None` discovers every published league. Awarded games are excluded. Defaults fit **one fixed Lasso**; set `SEARCH_GRID=ALPHA_GRID` to tune only on an inner 22/23 → 23/24 split.

CPU execution is appropriate for sklearn Lasso. `N_JOBS` counts independent folds, so this one-fold holdout cannot use several fold workers; native CPU threads are configured separately.

In [ ]:
DATA_ROOT = ROOT / "data" / "xDiyo_data"
OUTPUT_ROOT = ROOT / "experiments" / "total_corners_lasso"
EXPERIMENT_NAME = "Total corners | Lasso | train22-24 test24-25"
SEASONS = ("22_23", "23_24", "24_25")
LEAGUES = None
INCLUDE_AWARDED = False
INCLUDE_STANDINGS = True

LOOKBACK = 5
LONG_LOOKBACK = 10
MIN_PERIODS = 1
STD_DDOF = 1
EMA_SPAN = 5
CUTOFF_HOURS = None  # e.g. 48 predicts using information from two days earlier
AVAILABLE_AT = None  # optional history-aligned result-availability timestamps

USE_CORNER_RATINGS = False
USE_FEATURE_WARMUP = False
USE_RATING_TRANSITIONS = False
USE_TOP_K = False
TOP_K = 12
TOP_K_METHOD = "spearman"

LASSO_ALPHA = 0.1
LASSO_OPTIONS = dict(fit_intercept=True, precompute=False, copy_X=True,
                     max_iter=20_000, tol=1e-4, warm_start=False,
                     positive=False, random_state=42, selection="cyclic")
ALPHA_GRID = {"alpha": [0.01, 0.05, 0.1, 0.5, 1.0]}
SEARCH_GRID = None
SELECTION_METRIC = "mse"

N_JOBS = 1
INNER_THREADS = 1
REUSE_RESULTS = True
FINAL_REFIT = False  # opt in: an additional deployment fit on all three seasons
SAVE_FITTED_MODEL = True
RUN_FUTURE_PREDICTION = False
PREDICTION_ROUNDS = None  # e.g. {"Premier_League": [12, 13]}
PREDICTION_AS_OF = None

# These stages are intentionally disabled for this native Lasso baseline.
TRAINING_CONTROL = None  # no iterative callback / early-stopping support in Lasso.fit
TRAINING_VALIDATION = None  # alpha tuning uses inner folds instead
CHECKPOINT_POLICY = None  # completed runs/trials recover; Lasso has no mid-fit checkpoint
BET_ODDS = None  # supply real odds and an explicit bet decision policy before adding bets

## 2. Load seasons and select raw inputs
The quick table shows the exact league-season population. Only total corners and optional pregame standings are needed for this baseline; shots remain a separate table.

In [ ]:
loaded = load_seasons(
    DATA_ROOT, SEASONS, leagues=LEAGUES,
    tables=("matches", "statistics", "pregame"),
    include_awarded=INCLUDE_AWARDED, verify_hashes=False, record_dir=None,
)
source_summary = (loaded.matches.groupby(["source_league", "source_season"], dropna=False)
                  .size().rename("matches").reset_index())
display(source_summary)

corners = Stat("ALL", "Match overview", "cornerKicks", field="value")
selected_data = select_stats(
    loaded, bundles=("standings",) if INCLUDE_STANDINGS else (),
    stats=[("ALL", "Match overview", "cornerKicks")], categories=None,
)
team_catalog = TeamCatalog.from_matches(loaded.matches)

## 3. Prepare team histories, ratings and warm-up
Histories contain both team perspectives and normalized UTC kickoff timestamps. Rolling features below cross seasons within each team and league. Every historical operator excludes the match being predicted.

`None` cutoffs use earlier finished kickoffs as the documented retrospective assumption. A fixed offset controls prediction lead time. Optional corner Glicko rates the **comparison** of teams' corner counts, not the size of the corner margin. Warm-up wraps only the explicitly selected rolling feature; it is off by default.

In [ ]:
history = build_team_history(selected_data, stat_fields=("value",))
cutoffs = (None if CUTOFF_HOURS is None else
           history["kickoff_at"] - pd.Timedelta(hours=CUTOFF_HOURS))

warmup_policy = SeededEMA(alpha=0.5, handoff=Hard(rounds=1), league_weight=0.5,
                          round_keys=("competition_id", "season_id", "round"))
rating_engine = Glicko2(initial_rating=1500., initial_rd=350., initial_sigma=0.06,
                        tau=0.5, tolerance=1e-6)
rating_transition = GlickoTransition(
    phi_scale=1.15, movement_phi_scale=1.0, shrinkage=0.5,
    top=3, bottom=3, rank_by="standings", aggregate="mean",
) if USE_RATING_TRANSITIONS else None
rating_runs = {}
if USE_CORNER_RATINGS:
    rating_runs["corners"] = build_ratings(
        history, stat=corners, engine=rating_engine, higher_is_better=True,
        scope=("competition_id",), available_at=AVAILABLE_AT,
        transition=rating_transition, team_seasons=None, season_starts=None,
    )
print(f"{len(history):,} team-match history rows")

## 4. Define and evaluate predictors
`ForAgainst` selects the team's own corners or those conceded. Standard deviation and Z-score use individual match observations with `ddof=1`; the latest eligible observation belongs to the Z-score window. Partial windows use available observations, without padding.

The unobserved current match's corners are used **only as the label**, never as a raw input feature.

In [ ]:
corner_features = {}
for side in ("for", "against"):
    source = ForAgainst(corners, side=side)
    mean = RollingMean(source, window=LOOKBACK, min_periods=MIN_PERIODS)
    if USE_FEATURE_WARMUP:
        mean = WarmStart(mean, policy=warmup_policy)
    corner_features.update({
        f"corners_{side}_lag1": Lag(source, periods=1),
        f"corners_{side}_mean{LOOKBACK}": mean,
        f"corners_{side}_mean{LONG_LOOKBACK}": RollingMean(
            source, window=LONG_LOOKBACK, min_periods=MIN_PERIODS),
        f"corners_{side}_std{LOOKBACK}": RollingStd(
            source, window=LOOKBACK, min_periods=MIN_PERIODS, ddof=STD_DDOF),
        f"corners_{side}_z{LOOKBACK}": RollingZScore(
            source, window=LOOKBACK, min_periods=MIN_PERIODS, ddof=STD_DDOF),
        f"corners_{side}_ema": EMA(source, span=EMA_SPAN, min_periods=MIN_PERIODS),
    })
if INCLUDE_STANDINGS:
    corner_features["standing"] = NormalizedStanding(side="for", missing_value=0.)
if USE_CORNER_RATINGS:
    corner_features["corner_rating"] = Rating("corners", side="for", fields=("rating", "rd"))

feature_values = evaluate_features(
    history, corner_features, group_by=("team_id", "competition_id"),
    cutoffs=cutoffs, available_at=AVAILABLE_AT, team_counts=None,
    ratings=rating_runs or None, team_seasons=None, season_starts=None, keyed=True,
)
print(f"Defined {len(corner_features)} feature expressions")

## 5. Create total-corner labels and assemble one row per match
Assembly prefixes inputs with `home::` and `away::`, aligns on match/team identities, and removes rows whose total-corner label is missing. Imputation is deliberately deferred until each model's training fold.

In [ ]:
label_definitions = {"total_corners": MatchTotal(corners)}
labels = create_labels(history, label_definitions)
dataset = assemble_dataset(
    feature_values, labels["total_corners"], layout="match",
    feature_columns=None, target_columns=None, drop_missing_targets=True,
)
print(f"{len(dataset.X):,} labelled matches; {dataset.X.shape[1]} input columns")
display(pd.concat([dataset.metadata[["source_league", "source_season"]], dataset.y], axis=1).head())

## 6. Declare temporal holdout and optional inner selection folds
The **pooled calendar** uses `source_season`: provider `season_id` differs by league. One outer fold pools all eligible leagues, trains on 22/23–23/24, and tests 24/25. Any training match occurring after the earliest test cutoff is excluded by the splitter.

With optional search, only the 22/23 → 23/24 fold is used to choose alpha. 24/25 never participates in model selection. Feature histories may incorporate earlier completed test-season matches for later predictions; the fitted coefficients stay fixed for the full test season.

`AVAILABLE_AT` above is aligned to team-history rows. The splitter currently uses its kickoff proxy (`available_at=None`); to supply explicit split availability, pass a separate timestamp vector aligned to the assembled match dataset, joined by its match identity.

In [ ]:
split_cutoffs = (None if CUTOFF_HOURS is None else
                 dataset.metadata["kickoff_at"] - pd.Timedelta(hours=CUTOFF_HOURS))
outer_scheme = TemporalSplit(
    train_size=2, test_size=1, step=1, window="expanding", unit="seasons",
    calendar_by=(), block_by=("source_season",), gap=0, gap_time=None,
    score_start=0, score_rounds=None, allow_partial_test=False,
)
outer_plan = create_split_plan(dataset, outer_scheme, cutoffs=split_cutoffs, available_at=None)
inner_scheme = TemporalSplit(
    train_size=1, test_size=1, step=1, window="expanding", unit="seasons",
    calendar_by=(), block_by=("source_season",), gap=0, gap_time=None,
    score_start=0, score_rounds=None, allow_partial_test=False,
)
all_inner = create_split_plan(dataset, inner_scheme, cutoffs=split_cutoffs, available_at=None)
development_rows = set(outer_plan.folds[0].train.tolist())
inner_folds = [fold for fold in all_inner.folds
               if set(fold.train).issubset(development_rows)
               and set(fold.test).issubset(development_rows)]
selection_plan = SplitPlan(inner_folds, len(dataset.X), np.arange(len(dataset.X)))
fold_summary = pd.DataFrame([
    {"scope": scope, "fold": i, "train_matches": len(f.train),
     "test_matches": len(f.test), "scored_matches": len(f.score)}
    for scope, plan in (("outer evaluation", outer_plan), ("inner selection", selection_plan))
    for i, f in enumerate(plan.folds)
])
display(fold_summary)

prepared = PreparedExperiment(
    dataset, outer_plan, outputs={"source_summary": source_summary},
    config={"seasons": SEASONS, "leagues": LEAGUES, "awarded": INCLUDE_AWARDED,
            "prediction_lead_hours": CUTOFF_HOURS, "target": "total_corners",
            "layout": "match", "feature_warmup": USE_FEATURE_WARMUP,
            "corner_ratings": USE_CORNER_RATINGS},
)

## 7. Configure pre-training analysis and optional fitted feature selection
Descriptive studies use training rows only. If enabled, the top-k selector is fitted separately inside every actual fitting population, including inner search and final refit; it never learns from the test labels.

In [ ]:
preview_features = [f"home::corners_for_mean{LOOKBACK}", f"away::corners_for_mean{LOOKBACK}"]
pre_analysis = PreTrainingAnalysis({
    "Training distributions": FeatureDistributionReporter(
        type="per_fold", partition="train", features=preview_features,
        bins="auto", bandwidth="scott", grid_size=128),
    "Training correlations": CorrelationAnalysis(
        type="per_fold", partition="train", features=None, targets=None,
        methods=("pearson", "spearman", "kendall"), percentile_ranks=True),
    "Training feature timeline": FeatureTimeline(
        type="timeline", partition="train", features=preview_features,
        entity="team", teams=None, team_column=None, max_points=1000),
})
fitted_analysis = PreTrainingAnalysis({
    "Top correlation inputs": TopKCorrelationSelector(
        type="per_fold", partition="train", k=TOP_K, method=TOP_K_METHOD,
        source=None, features=None, targets=None, across_folds=False),
}) if USE_TOP_K else None

## 8. Create a fresh Lasso pipeline for every fit
Each fit learns **median imputation → standardization → Lasso** exclusively on its training population. All-empty training columns are retained by the imputer so coefficient names stay aligned. No previously fitted estimator is reused across candidates or folds.

Coefficients describe standardized inputs; correlated features can exchange weight. Lasso is an unconstrained count baseline and can return negative or fractional predictions. This notebook does not clip or round them silently, and a predicted total is not an over/under bet probability.

In [ ]:
def make_lasso_adapter(alpha):
    return EstimatorAdapter(Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True,
                                  add_indicator=False)),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("lasso", Lasso(alpha=alpha, **LASSO_OPTIONS)),
    ]), prediction_methods=("predict",))


def make_candidate(parameters):
    alpha = parameters["alpha"]
    return Candidate(
        name="Lasso", model_factory=partial(make_lasso_adapter, alpha),
        config={"alpha": alpha, "model": "Lasso", "lasso": LASSO_OPTIONS,
                "preprocessing": "median -> StandardScaler", "top_k": TOP_K if USE_TOP_K else None},
        feature_columns=None, target_columns=None,
        pre_analysis=fitted_analysis,
        features_from="Top correlation inputs" if USE_TOP_K else None,
        control=TRAINING_CONTROL, validation=TRAINING_VALIDATION,
        observer=None, complexity=None, fit_statistics=None,
    )

fixed_candidate = make_candidate({"alpha": LASSO_ALPHA})
model_selection = (None if SEARCH_GRID is None else ModelSelection(
    GridCandidates(SEARCH_GRID, make_candidate), metrics=(SELECTION_METRIC,),
    decision=None, pooling=None, information_criteria=False, on_error="raise",
    evidence_reporters={},
))
execution = ExecutionPolicy(n_jobs=N_JOBS, device="cpu", inner_threads=INNER_THREADS, gpu_jobs=1)
refit_policy = (RefitPolicy(train_positions=np.arange(len(dataset.X)), candidate=None,
                           validation=None, control=None) if FINAL_REFIT else None)

## 9. Configure post-training studies
The report includes errors, observed/predicted KDE curves, residuals, match results with team names, and surviving Lasso coefficients. A two-corner tolerance only controls match-row coloring; it does not alter metrics.

The leaderboard pools saved **final runs in this named experiment**, including earlier sessions. Its normalized 50/50 weights rank MSE and MAE after direction-aware percentile scaling. Internal search trials stay out of the leaderboard. Betting is not run because no real decimal odds or bet policy were supplied.

In [ ]:
leaderboard_weights = {"mse": 0.5, "mae": 0.5}
leaderboard_selectors = {
    metric: {"metric": metric, "study": "Overall errors"}
    for metric in leaderboard_weights
}
post_analysis = PostTrainingAnalysis({
    "Overall errors": PerformanceReporter(type="overall", partition="score",
                                          metrics=("mse", "mae", "r2")),
    "Observed vs predicted": PredictionDistributionReporter(
        type="overall", partition="score", mode="kde", bandwidth="scott", grid_size=128),
    "Residuals": ResidualAnalysisReporter(type="overall", partition="score", bins="auto"),
    "Prediction timeline": PredictionTimelineReporter(type="timeline", partition="test",
                                                     group_by="source_league"),
    "Match results": MatchResultReporter(
        type="overall", partition="score", comparison="numeric", tolerance=2.,
        league=None, season=None, team=None, round=None,
        league_column="source_league", season_column="source_season",
        catalog=team_catalog, show_badges=False, page_size=25),
    "Surviving Lasso coefficients": CoefficientReporter(
        type="per_fold", partition="model", tolerance=0., include_zeros=False,
        survival_frequency=True),
    "Saved experiment leaderboard": ExperimentLeaderboardReporter(
        weights=leaderboard_weights, selectors=leaderboard_selectors,
        scaling="percentile", reference_scales={}, directions={}, include_trials=False),
})

## 10. Run, save artifacts and show the report
An exact completed run can be reused without fitting. Completed grid trials are recoverable after interruption; the current Lasso fit itself is not checkpointed. Set `REUSE_RESULTS=False` for an explicitly fresh fit.

When `FINAL_REFIT=True`, an additional deployment model is fitted on all three seasons **after evaluation**. Its predictions are not substituted into the test report.

In [ ]:
experiment = FootballExperiment(EXPERIMENT_NAME, output_dir=OUTPUT_ROOT)
run_options = dict(
    pre_analysis=pre_analysis, post_analysis=post_analysis,
    refit_policy=refit_policy, checkpoint_policy=CHECKPOINT_POLICY,
    execution=execution, name_fields=["alpha"], reuse=REUSE_RESULTS,
)
if model_selection is None:
    run_options["model"] = fixed_candidate
else:
    if not selection_plan.folds:
        raise ValueError("No inner fold fits entirely inside the outer training population.")
    run_options.update(model_selection=model_selection, selection_plan=selection_plan,
                       development_positions=outer_plan.folds[0].train)
result = experiment.run(prepared, **run_options)
print("Reused:" if result.reused else "Completed:", result.record["name"])
print("Artifacts:", result.path)
result.show()

## 11. Inspect predictions and retain a model
Saved result bundles retain numerical results and reports; fitted model persistence is explicit. The model directory belongs to this run. On reuse, an existing model is loaded; if it was never saved, the notebook explains how to request a fresh fit rather than silently retraining.

In [ ]:
display(result.training.prediction_frame().head(10))
html_path = Path(result.path) / "notebook_report.html"
result.to_html(html_path)

fitted_model = None
model_path = Path(result.path) / ("deployment_model" if FINAL_REFIT else "evaluation_model")
if SAVE_FITTED_MODEL:
    if model_path.exists():
        fitted_model = load_model(model_path)
    elif result.reused:
        print("This reused result has no saved model. Set REUSE_RESULTS=False and rerun to fit and save one.")
    else:
        save_model(result.refit if FINAL_REFIT else result.training, model_path)
        fitted_model = load_model(model_path)
    if fitted_model is not None:
        print("Saved model loaded:", model_path)
print("Offline report:", html_path)

## 12. Optional prediction for published upcoming rounds
The saved model consumes the same feature definitions and ordering. If the exports include unplayed fixtures, set `RUN_FUTURE_PREDICTION=True`; optionally choose league-specific rounds. All history remains loaded when narrowing the prediction rows. Include a live season in the data configuration before using this with newer fixtures, and revise the evaluation season design separately.

Only load trusted model artifacts you created. The model includes fitted preprocessing; loading it does not fit anything.

In [ ]:
if RUN_FUTURE_PREDICTION:
    if fitted_model is None:
        raise ValueError("Save/load a fitted model in the preceding cell first.")
    fixtures = select_prediction_fixtures(
        loaded, statuses=("notstarted",), rounds=PREDICTION_ROUNDS, as_of=PREDICTION_AS_OF)
    if len(fixtures.fixtures) == 0:
        print("No upcoming fixtures match these filters in the loaded exports.")
    else:
        future_dataset = assemble_dataset(feature_values, labels["total_corners"],
                                           layout="match", drop_missing_targets=False)
        aligned = fixtures.align(future_dataset)
        future_predictions = fitted_model.predict(aligned)
        display(pd.concat([aligned.metadata, future_predictions["predict"]], axis=1).head(30))
else:
    print("Future prediction disabled; evaluation predictions are retained above.")

## Next steps
- Inspect match errors, coefficient survival and train-only feature studies before changing the baseline.
- Enable the alpha grid for a separate run, or change the lookbacks and give the experiment a descriptive name.
- Keep 24/25 as the final evaluation population while tuning on the earlier seasons. Repeated manual choices informed by 24/25 turn it into exploration; use a new untouched season for a later final assessment.
- [Pipeline configuration and validation notes](../docs/analytics/lasso_total_corners.md).